In [1]:

import numpy as np
import pandas as pd

from coin_flip_with_riskless_asset_model import CoinFlipWithRisklessAssetModel

from finlib.ensemble_of_returns_paths import EnsembleOfReturnsPaths

In [2]:
coin_flip_model = CoinFlipWithRisklessAssetModel(gamma_heads=2.0, alpha=1.25, r=0.97, p=0.5)

In [3]:
seed = 12345
rng = np.random.default_rng(seed)

In [4]:
gross_returns_tensor = coin_flip_model.generate_random_gross_returns(rng, num_periods=10, num_paths=5)


In [ ]:
"""Plot-preparation helpers
 
These assemble the trusted per-period summaries produced by
``ReturnsPath.summarize_across_paths`` (in finlib) into the exact tidy
frames each bespoke Plotly figure wants. The statistical claims live and
are tested in finlib; nothing here recomputes them.
"""
 
def growth_rate_summary(paths: EnsembleOfReturnsPaths) -> pd.DataFrame:
    """Per-period summary of the running growth rate across paths.
 
    Threshold 0.0: a path is "above" once its running growth rate is
    positive. Index is the period, 1..n.
    """
    return paths.summarize_across_paths(paths.running_growth_rate, threshold=0.0)
 
 
def wealth_summary(paths: EnsembleOfReturnsPaths) -> pd.DataFrame:
    """Per-period summary of wealth (relative to W_0) across paths.
 
    Threshold 1.0: break-even wealth is a gross of 1, not 0. Reads
    ``running_wealth`` off the object rather than recomputing exp(g*t),
    so the two cannot drift; index is the period, 0..n (running_wealth
    includes W_0).
    """
    wealth_relative = paths.running_wealth / np.asarray(paths.w0)[..., None] \
        if np.ndim(paths.w0) else paths.running_wealth / paths.w0
    return paths.summarize_across_paths(wealth_relative, threshold=1.0)

In [ ]:




# GENERATE DATA FOR THE PLOT OF THE ARITHMETIC VS GEOMETRIC RETURN AS A FUNCTION OF ALLOCATION TO THE RISKY ASSET

def generate_arithmetic_vs_geometric_data(params: CoinFlipWithRisklessAssetModel) -> pd.DataFrame:

    output = []
    for f in np.linspace(0, 1, 1001):
        weights_vector = np.array([1 - f, f])
        arithmetic_gross_return = params.return_arithmetic_portfolio_gross_return(weights_vector) # E[Y] = weights_vector . E[X]
        geometric_gross_return = params.return_geometric_portfolio_gross_return(weights_vector) # exp(E[log(Y)])
        growth_rate = params.return_expected_log_portfolio_gross_return(weights_vector) # E[log(Y)]
        growth_rate_ceiling = np.log(params.return_arithmetic_portfolio_gross_return(weights_vector)) # log(E[Y]). This follows from Jensen's inequality.

        output.append((f, arithmetic_gross_return, geometric_gross_return, growth_rate, growth_rate_ceiling))
    df_arith_geo = pd.DataFrame(output, columns=['f', 'Arithmetic Gross Return', 'Geometric Gross Return', 'Growth Rate', 'Growth Rate Ceiling'])
    return df_arith_geo

In [ ]:
f_star = opt_result_for_CRP.x

weights_for_risky_asset = np.array([0.0, 1.0])
weights_for_optimal_CRP = np.array([1 - f_star, f_star])
num_simulations = 200
size = 15 # number of periods to simulate

In [ ]:
running_empirical_growth_rates = simulate_running_empirical_growth_rates(rng, params, weights_for_optimal_CRP, num_simulations, size)

df_risky_asset_for_wealth = return_wealth_over_time_dataframe_for_plotting(running_empirical_growth_rates)
df_optimal_CRP_for_wealth = return_wealth_over_time_dataframe_for_plotting(running_empirical_growth_rates)

fig_risky_asset_for_wealth = create_wealth_over_time_plot(df_risky_asset_for_wealth, title="Simulating the Risky Asset")
fig_optimal_CRP_for_wealth = create_wealth_over_time_plot(df_optimal_CRP_for_wealth, title="Simulating the Optimal CRP")

In [ ]:
df_risky_asset_for_growth_rate = return_empirical_growth_rates_dataframe_for_plotting(running_empirical_growth_rates)
df_optimal_CRP_for_growth_rate = return_empirical_growth_rates_dataframe_for_plotting(running_empirical_growth_rates)

fig_risky_asset_for_growth_rate = create_empirical_growth_rates_plot(df_risky_asset_for_growth_rate, weights_for_risky_asset, params, title="Simulating the Risky Asset")
fig_optimal_CRP_for_growth_rate = create_empirical_growth_rates_plot(df_optimal_CRP_for_growth_rate, weights_for_optimal_CRP, params, title="Simulating the Optimal CRP")

In [ ]:
df_arith_geo = generate_arithmetic_vs_geometric_data(params)
fig_arith_geo = generate_arithmetic_vs_geometric_plot(df_arith_geo, opt_result_for_CRP)

In [ ]:
# Set the parameters for the model

params = CoinFlipWithRisklessAssetModel(gamma_heads=2.0, p=0.5, alpha=1.25, r=0.97)
